# RAFT-Net demo: real forgery localisation with a trained checkpoint

Loads one finished model and shows what it actually localises:
0. **Segment one image:** input | predicted mask | probability | overlay, saved as PNG files.
1. **Test-set showcase:** for each attached test set, 40 random tampered images are scored; the notebook shows the
   **best, typical and worst** cases (image | ground truth | probability heatmap | overlay, with per-image F1 / IoU).
   Selection is automatic, so failures are shown as honestly as successes.
2. **Robustness demo:** the same images clean, after JPEG quality 90 / 70 / 50, and after the social-network (OSN) chain.
3. **Your own images:** attach any folder of images as a dataset (name containing `demo`), or set `USER_IMAGE_DIR`;
   predictions are shown without ground truth.

4. **Efficiency:** parameters, inference latency and peak GPU memory of ForMa and RAFT-Net variants (512×512, batch 1).

All figures are saved to `/kaggle/working/demo/` for the report.

**Inputs:** the training notebook that holds the checkpoint (e.g. `raftnet_fullrun6` for `raftnet_nofcss_s1222`,
or the 30 h account's notebook for `raftnet_lite_s1222`), plus any of: CASIA v2, CASIA v1 (`casia dataset` +
`casia1-groundtruth` + `casiav1-renamed`), Columbia, IMD2020. **Settings:** GPU, Internet on. Runtime ≈ 10 minutes.


## 1. Setup

In [ ]:
import subprocess, sys, os

REPO_DIR = '/kaggle/working/ForMa_repo'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/multimediaFor/ForMa.git', REPO_DIR], check=True)

MODELS_DIR = os.path.join(REPO_DIR, 'models')
print('Repo cloned at:', REPO_DIR)
print('Models dir:', MODELS_DIR, '| exists:', os.path.isdir(MODELS_DIR))


# -- Fix: ForMa's own csm_triton.py has an unguarded @triton.jit decorator --------------
try:
    import triton
    print("triton already available:", triton.__version__)
except ImportError:
    print("triton not installed -- attempting pip install...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'triton'],
                    capture_output=True, text=True)
    try:
        import triton
        print("triton installed successfully:", triton.__version__)
    except ImportError:
        print("triton still unavailable -- injecting a no-op stub so csm_triton.py can import "
              "without crashing. Safe: WITH_TRITON correctly evaluates False in that file, so "
              "the actual computation already uses the verified pure-PyTorch scan regardless "
              "of this stub.")
        import types
        _triton_stub = types.ModuleType('triton')
        _triton_stub.jit = lambda fn=None, **kwargs: (fn if fn is not None else (lambda f: f))
        _tl_stub = types.ModuleType('triton.language')
        _tl_stub.constexpr = object
        _triton_stub.language = _tl_stub
        sys.modules['triton'] = _triton_stub
        sys.modules['triton.language'] = _tl_stub


In [ ]:
import importlib, subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

for pkg, imp in [('timm', 'timm'), ('fvcore', 'fvcore'), ('einops', 'einops'), ('opencv-python-headless', 'cv2')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        pip_install(pkg)

# mamba-ssm / causal-conv1d: ForMa's own code (csms6s.py) gracefully falls back to a
# pure-PyTorch selective-scan if no CUDA kernel is importable -- a failed install here
# costs speed, not correctness.
try:
    import selective_scan_cuda  # noqa: F401
    print("mamba-ssm's selective_scan_cuda already importable.")
except ImportError:
    print("Installing mamba-ssm + causal-conv1d (compiles CUDA kernels, can take a few minutes)...")
    try:
        pip_install('causal-conv1d>=1.2.0', 'mamba-ssm', '--no-build-isolation')
        import selective_scan_cuda  # noqa: F401
        print("mamba-ssm installed and selective_scan_cuda is importable.")
    except Exception as e:
        print("mamba-ssm install/import did not succeed:", e)
        print("Not fatal for THIS model -- falls back to a pure-PyTorch scan (slower, correct).")


In [ ]:
vmamba_path = os.path.join(MODELS_DIR, 'vmamba_pixelshuf_modals.py')
with open(vmamba_path, 'r') as f:
    src = f.read()

patch1_old = '''with open('../config_pixelshuf_modals.yaml', 'r') as f:
    args = yaml.safe_load(f)
device = get_device(args["cuda_idx"])'''
patch1_new = '''args = {"cuda_idx": 0}  # only args["cuda_idx"] is referenced elsewhere in this file
device = get_device(0)'''

patch2_old = '''        with torch.cuda.device(args["cuda_idx"]):
            y = self.forward_core(x)'''
patch2_new = '''        if torch.cuda.is_available():
            with torch.cuda.device(args["cuda_idx"]):
                y = self.forward_core(x)
        else:
            y = self.forward_core(x)'''

# Patch 3: SS2D's default forward_type "v2" hardcodes selective_scan_backend="oflex" --
# VMamba's own separate custom CUDA kernel, never installed here. mamba-ssm's kernel
# (backend="mamba") DID import successfully -- swapping the string is a pure bugfix.
patch3_old = '''            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="oflex"),#core
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="oflex"),'''
patch3_new = '''            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="mamba"),#core -- patched: oflex kernel unavailable, mamba backend verified working
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="mamba"),  # patched, same reason'''

# Patch 5: a leftover debug print fires on every forward pass, flooding the log.
patch5_old = '''        x = self.patch_embed(x)#[4, 128, 128, 96]
        print(x.shape,"=================")'''
patch5_new = '''        x = self.patch_embed(x)#[4, 128, 128, 96]'''

n_patches = 0
if patch1_old in src:
    src = src.replace(patch1_old, patch1_new); n_patches += 1
if patch2_old in src:
    src = src.replace(patch2_old, patch2_new); n_patches += 1
if patch3_old in src:
    src = src.replace(patch3_old, patch3_new); n_patches += 1
if patch5_old in src:
    src = src.replace(patch5_old, patch5_new); n_patches += 1

with open(vmamba_path, 'w') as f:
    f.write(src)

print(f"Applied {n_patches}/4 patches to {vmamba_path}")
assert n_patches == 4, "Expected patch text not found -- ForMa's repo may have changed upstream."

# Patch 4: dispatch-fallback safety net in csms6s.py, defense in depth beyond patch 3.
csms6s_path = os.path.join(MODELS_DIR, 'csms6s.py')
with open(csms6s_path, 'r') as f:
    csms6s_src = f.read()

patch4_old = '''    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    fn = selective_scan_torch if backend == "torch" or (not WITH_CUDA) else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)'''
patch4_new = '''    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    _avail = {"oflex": WITH_SELECTIVESCAN_OFLEX, "core": WITH_SELECTIVESCAN_CORE, "mamba": WITH_SELECTIVESCAN_MAMBA}
    if backend == "torch" or not WITH_CUDA:
        backend = "torch"
    elif backend not in _avail or not _avail.get(backend, False):
        backend = next((b for b in ("mamba", "core", "oflex") if _avail[b]), "torch")
    fn = selective_scan_torch if backend == "torch" else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)'''

n_patches_4 = 0
if patch4_old in csms6s_src:
    csms6s_src = csms6s_src.replace(patch4_old, patch4_new)
    n_patches_4 = 1

with open(csms6s_path, 'w') as f:
    f.write(csms6s_src)

print(f"Applied {n_patches_4}/1 dispatch-fallback patch to {csms6s_path}")
assert n_patches_4 == 1, "Expected patch text not found in csms6s.py -- ForMa's repo may have changed upstream."


In [ ]:
# ── FCSS: patch VMamba's selective scan so a per-position forensic bias can steer the step size ──
# Must run BEFORE vmamba_pixelshuf_modals is imported (each SS2D binds its scan function at construction).
# A model that never sets `forensic_bias` (the ForMa baseline) runs exactly as before.
_vm = os.path.join(MODELS_DIR, 'vmamba_pixelshuf_modals.py')
_src = open(_vm).read()
_anchor = "            xs = xs.view(B, -1, L)\n            dts = dts.contiguous().view(B, -1, L)\n"
_inject = _anchor + (
    "            _fb = getattr(self, 'forensic_bias', None)  # RAFT-Net FCSS\n"
    "            if _fb is not None:\n"
    "                if tuple(_fb.shape[-2:]) != (H, W):\n"
    "                    _fb = F.interpolate(_fb, size=(H, W), mode='bilinear', align_corners=False)\n"
    "                _fbs = cross_scan_fn(_fb.to(x.dtype).contiguous(), in_channel_first=True, out_channel_first=True,\n"
    "                                     scans=_scan_mode, force_torch=scan_force_torch)\n"
    "                dts = dts + _fbs.contiguous().view(B, -1, L).to(dts.dtype)\n")
if 'RAFT-Net FCSS' in _src:
    print('FCSS patch already present')
else:
    assert _src.count(_anchor) == 1, 'FCSS anchor not found exactly once -- ForMa code changed?'
    open(_vm, 'w').write(_src.replace(_anchor, _inject))
    print('FCSS patch applied to', _vm)


## 2. Configuration

In [ ]:
import os, sys, glob, json, random
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

JOB = 'raftnet_lite_nofullres_s1223'   # final RAFT-Net (best seed by validation F1)
                               # after RAFT-Net-lite finishes: 'raftnet_lite_s1222'
N_SAMPLE = 40                  # random tampered images scored per test set
SHOW = {'best': 2, 'typical': 2, 'worst': 2}
ROBUST_IMAGES = 3
USER_IMAGE_DIR = None          # e.g. '/kaggle/input/my-demo-images'; auto-detected if a dataset name contains 'demo'
SEED, INPUT, THR = 1221, 512, 0.5
OUT = '/kaggle/working/demo'
os.makedirs(OUT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
random.seed(SEED); np.random.seed(SEED)
print('device:', device)


## 3. Load the trained model

In [ ]:
MODEL_SRC = r'''
# raftnet.py -- RAFT-Net and the official ForMa baseline, built on ForMa's own classes.
import torch
import torch.nn as nn
import torch.nn.functional as F
from vmamba_pixelshuf_modals import (Backbone_VSSM, SegFormerHead, ModalMixer, ModalitiesExtractor, ConvModule)

IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD = (0.229, 0.224, 0.225)


class Denorm(nn.Module):
    """Model input is ImageNet-normalised (for the backbone); forensic extractors need raw [0,1] pixels
    (Noiseprint++ was trained on [0,1] RGB). Applied identically in ForMa and RAFT-Net."""
    def __init__(self):
        super().__init__()
        self.register_buffer('mean', torch.tensor(IMNET_MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(IMNET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        return x * self.std + self.mean


def conv_bn_relu(cin, cout, k=3, s=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, k // 2, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


# ── Official ForMa baseline ───────────────────────────────────────────────────────────────────
class ForMaOfficial(nn.Module):
    """Forensic_Vmamba as released: VMamba-T backbone; Noiseprint++ (frozen) + Bayar + SRM on the image
    shrunk to 1/4, mixed to 3 channels, fed to the decoder only."""
    def __init__(self, np_weights):
        super().__init__()
        self.denorm = Denorm()
        self.backbone = Backbone_VSSM()
        self.decode = SegFormerHead()
        self.modalmixer = ModalMixer()
        self.modal_extractor = ModalitiesExtractor(['noiseprint', 'bayar', 'srm'], np_weights)

    def train(self, mode=True):
        super().train(mode)
        self.modal_extractor.noiseprint.eval()     # ForMa's model.train() would silently re-enable BN updates
        return self

    def forward(self, x, dct=None, return_feat=False):
        n, _, h, w = x.shape
        y = self.backbone(x)
        xm = F.interpolate(self.denorm(x), size=(h // 4, w // 4), mode='bilinear', align_corners=False)
        xm = self.modalmixer(self.modal_extractor.multi_output(xm))
        return {'mask': self.decode(y, xm)}


class ForMaWide(ForMaOfficial):
    """Official ForMa with ONE change: the width w of its forensic pathway. ForMa's mixer compresses 96 channels of
    noise features to 3 (then linearly expands them to 16 before fusion with 384 RGB channels). Here the mixer outputs
    w channels and w channels enter the fusion layer; backbone, extractors, mixer blocks and decoder are unchanged."""
    def __init__(self, np_weights, width):
        super().__init__(np_weights)
        self.modalmixer = ModalMixer(out_channels=width)
        self.decode.conv_modals = nn.Conv2d(width, width, kernel_size=1)
        self.decode.linear_fuse = ConvModule(c1=96 * 4 + width, c2=96, k=1)


# ── RAFT-Net components ───────────────────────────────────────────────────────────────────────
class ForensicStem(nn.Module):
    """Forensic evidence at every backbone scale (1/4, 1/8, 1/16, 1/32).
    fullres=True : SRM + Bayar residuals computed on the FULL-resolution image, downsampled by learned
                   stride-2 convolutions (ForMa computes them on a bilinearly shrunk image, which blurs away
                   the fine residuals these filters exist to capture).
    Noiseprint++ : frozen camera-fingerprint net, at 1/4 as in ForMa (full resolution is ~16x too slow on a T4).
    dct=True     : native-grid block-DCT map (computed in the data loader on the original-resolution image,
                   JPEG grid intact) injected at 1/8."""
    def __init__(self, np_weights, fullres=True, dct=True, widths=(64, 128, 192, 256)):
        super().__init__()
        self.fullres, self.use_dct, self.widths = fullres, dct, widths
        self.hp = ModalitiesExtractor(['bayar', 'srm'])
        self.npp = ModalitiesExtractor(['noiseprint'], np_weights)
        self.hp_down = (nn.Sequential(conv_bn_relu(6, 32, 3, 2), conv_bn_relu(32, 48, 3, 2)) if fullres
                        else conv_bn_relu(6, 48, 3, 1))
        self.np_conv = conv_bn_relu(3, 16, 3, 1)
        self.f4 = conv_bn_relu(48 + 16, widths[0], 3, 1)
        self.d8 = conv_bn_relu(widths[0], widths[1], 3, 2)
        if dct:
            self.dct_enc = nn.Sequential(conv_bn_relu(64, widths[1], 1, 1), conv_bn_relu(widths[1], widths[1], 3, 1))
        self.d16 = conv_bn_relu(widths[1], widths[2], 3, 2)
        self.d32 = conv_bn_relu(widths[2], widths[3], 3, 2)

    def forward(self, x01, dct=None):
        h, w = x01.shape[-2:]
        q = F.interpolate(x01, size=(h // 4, w // 4), mode='bilinear', align_corners=False)
        hp = torch.cat(self.hp.multi_output(x01 if self.fullres else q), dim=1)
        f4 = self.f4(torch.cat([self.hp_down(hp), self.np_conv(self.npp.multi_output(q)[0])], dim=1))
        f8 = self.d8(f4)
        if self.use_dct:
            if dct is None:
                raise ValueError('RAFT-Net with dct=True needs the native-grid DCT map as input')
            if dct.shape[-2:] != f8.shape[-2:]:
                dct = F.interpolate(dct, size=f8.shape[-2:], mode='nearest')
            f8 = f8 + self.dct_enc(dct.to(f8.dtype))
        f16 = self.d16(f8)
        return [f4, f8, f16, self.d32(f16)]


class FCSS(nn.Module):
    """Forensic-Conditioned Selective Scan. For every VMamba block, a ZERO-initialised 1x1 conv maps the stage's
    forensic features to a bias on the SSM step size (dt) at every position, bounded by SCALE*tanh(.). The patched
    SS2D.forward_corev2 adds it (cross-scanned into all 4 scan directions) before the selective scan. Zero init =>
    the network starts exactly as the ImageNet-pretrained VMamba; training learns how far forensic evidence
    should steer the scan."""
    SCALE = 2.0

    def __init__(self, backbone, in_widths):
        super().__init__()
        convs, self._ops = [], []
        for s, layer in enumerate(backbone.layers):
            for blk in layer.blocks:
                d_inner = blk.op.dt_projs_weight.shape[1]
                c = nn.Conv2d(in_widths[s], d_inner, 1)
                nn.init.zeros_(c.weight)
                nn.init.zeros_(c.bias)
                convs.append(c)
                self._ops.append((s, blk.op))
        self.convs = nn.ModuleList(convs)

    def set(self, feats):
        for (s, op), conv in zip(self._ops, self.convs):
            op.forensic_bias = self.SCALE * torch.tanh(conv(feats[s]))

    def clear(self):
        for _, op in self._ops:
            op.forensic_bias = None


class WideHead(SegFormerHead):
    """ForMa's pixel-shuffle decoder, with the forensic pathway widened (ForMa: 3 -> 16 channels next to 384 RGB
    channels). Same computation as SegFormerHead.forward; optionally also returns the forensic features (for ACR)."""
    def __init__(self, modal_in=64, modal_out=32, embedding_dim=96):
        super().__init__(embedding_dim=embedding_dim)
        self.conv_modals = nn.Conv2d(modal_in, modal_out, kernel_size=1)
        self.linear_fuse = ConvModule(c1=embedding_dim * 4 + modal_out, c2=embedding_dim, k=1)

    def forward(self, inputs, x_modals, return_feat=False):
        c1, c2, c3, c4 = inputs
        m = self.conv_modals(x_modals)
        n = c4.shape[0]
        _c4 = F.pixel_shuffle(self.linear_c4(c4).permute(0, 2, 1).reshape(n, -1, c4.shape[2], c4.shape[3]), 8)
        _c3 = F.pixel_shuffle(self.linear_c3(c3).permute(0, 2, 1).reshape(n, -1, c3.shape[2], c3.shape[3]), 4)
        _c2 = F.pixel_shuffle(self.linear_c2(c2).permute(0, 2, 1).reshape(n, -1, c2.shape[2], c2.shape[3]), 2)
        _c1 = self.linear_c1(c1).permute(0, 2, 1).reshape(n, -1, c1.shape[2], c1.shape[3])
        out = self.linear_pred(self.dropout(self.linear_fuse(torch.cat([_c4, _c3, _c2, _c1, m], dim=1))))
        return (out, m) if return_feat else out


class BoundaryModule(nn.Module):
    """BASNet-style boundary-motivated module (from the RAFT-Net prototype, unchanged)."""
    def __init__(self, c_low1=96, c_low2=192, c_high=768, mid_channels=64):
        super().__init__()
        self.conv_f1 = nn.Sequential(nn.Conv2d(c_low1, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))
        self.conv_f2 = nn.Sequential(nn.Conv2d(c_low2, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))
        self.gate = nn.Sequential(nn.Conv2d(mid_channels * 2, mid_channels, 1), nn.ReLU(inplace=True),
                                  nn.Conv2d(mid_channels, mid_channels, 1))
        self.attn = nn.Sequential(nn.Conv2d(mid_channels, mid_channels, 1), nn.Sigmoid())
        self.conv_high = nn.Sequential(nn.Conv2d(c_high, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))

    def forward(self, f1, f2, f_high):
        hw = f1.shape[-2:]
        F1 = self.conv_f1(f1)
        F2 = self.conv_f2(F.interpolate(f2, size=hw, mode='bilinear', align_corners=True))
        FL = self.attn(self.gate(torch.cat([F1, F2], dim=1)))
        FH = self.conv_high(F.interpolate(f_high, size=hw, mode='bilinear', align_corners=True))
        return FL * FH


class EdgeEnhance(nn.Module):
    """BASNet-style boundary-enhanced module (from the RAFT-Net prototype, unchanged)."""
    def __init__(self, channels=64):
        super().__init__()
        self.pool1 = nn.AdaptiveAvgPool2d(1)
        self.pool2 = nn.AdaptiveAvgPool2d(4)
        self.atrous = nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2)
        self.fuse = nn.Conv2d(channels * 2, channels, kernel_size=1)
        self.swish = nn.SiLU(inplace=True)

    def forward(self, x):
        h, w = x.shape[-2:]
        p1 = F.interpolate(self.pool1(x), size=(h, w), mode='bilinear', align_corners=True)
        p2 = F.interpolate(self.pool2(x), size=(h, w), mode='bilinear', align_corners=True)
        return x + self.swish(self.atrous(x)) * torch.sigmoid(self.fuse(torch.cat([p1, p2], dim=1)))


class BoundaryBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.bmm = BoundaryModule()
        self.bem = EdgeEnhance(64)
        self.head = nn.Sequential(conv_bn_relu(64, 32, 3, 1), nn.Conv2d(32, 1, 1))

    def forward(self, c1, c2, c4, out_hw):
        b = self.head(self.bem(self.bmm(c1, c2, c4)))
        return F.interpolate(b, size=out_hw, mode='bilinear', align_corners=True)


# ── RAFT-Net ──────────────────────────────────────────────────────────────────────────────────
class RAFTNet(nn.Module):
    def __init__(self, np_weights, fcss=True, dct=True, fullres=True, boundary=True):
        super().__init__()
        self.cfg = dict(fcss=fcss, dct=dct, fullres=fullres, boundary=boundary)
        self.denorm = Denorm()
        self.backbone = Backbone_VSSM()
        self.stem = ForensicStem(np_weights, fullres=fullres, dct=dct)
        self.fcss = FCSS(self.backbone, self.stem.widths) if fcss else None
        self.decode = WideHead(modal_in=self.stem.widths[0], modal_out=32)
        self.boundary = BoundaryBranch() if boundary else None

    def train(self, mode=True):
        super().train(mode)
        self.stem.npp.noiseprint.eval()
        return self

    def forward(self, x, dct=None, return_feat=False):
        h, w = x.shape[-2:]
        feats = self.stem(self.denorm(x), dct)
        if self.fcss is not None:
            self.fcss.set(feats)
        try:
            y = self.backbone(x)
        finally:
            if self.fcss is not None:
                self.fcss.clear()
        out = {}
        if return_feat:
            out['mask'], out['feat'] = self.decode(y, feats[0], return_feat=True)
        else:
            out['mask'] = self.decode(y, feats[0])
        if self.boundary is not None:
            out['boundary'] = self.boundary(y[0], y[1], y[3], (h, w))
        return out


VARIANTS = {
    # name                  builder kwargs (None = official ForMa)                                    needs DCT input
    'forma':               (None,                                                                     False),
    'raftnet':             (dict(fcss=True,  dct=False, fullres=True,  boundary=True),                False),  # FINAL
    'raftnet_no_fcss':     (dict(fcss=False, dct=False, fullres=True,  boundary=True),                False),
    'raftnet_no_boundary': (dict(fcss=True,  dct=False, fullres=True,  boundary=False),               False),
    'raftnet_no_fullres':  (dict(fcss=True,  dct=False, fullres=False, boundary=True),                False),
    'raftnet_lite_nofullres': (dict(fcss=False, dct=False, fullres=False, boundary=True),             False),  # final candidate
    'forma_w12':           (dict(width=12),                                                            False),  # bottleneck study
    'forma_w48':           (dict(width=48),                                                            False),  # bottleneck study
}


def build_model(variant, np_weights):
    kw, _ = VARIANTS[variant]
    if kw is None:
        return ForMaOfficial(np_weights)
    if 'width' in kw:
        return ForMaWide(np_weights, kw['width'])
    return RAFTNet(np_weights, **kw)


def load_imagenet_backbone(model, sd, prefix='backbone.'):
    """ForMa train.py logic: copy checkpoint tensors into model[prefix + name] when shapes match."""
    msd = model.state_dict()
    n = 0
    for k, v in sd.items():
        mk = prefix + k
        if mk in msd and tuple(msd[mk].shape) == tuple(v.shape):
            msd[mk] = v
            n += 1
    model.load_state_dict(msd)
    bb = [k for k in msd if k.startswith(prefix)]
    return n, len(sd), len(bb)
'''


In [ ]:
with open(os.path.join(MODELS_DIR, 'raftnet.py'), 'w') as f:
    f.write(MODEL_SRC)
if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)
import importlib, raftnet
importlib.reload(raftnet)
from raftnet import build_model
NP_WEIGHTS = os.path.join(REPO_DIR, 'weights', 'np++.pth')

def find_file(name, base='/kaggle/input'):
    for r, _, fs in os.walk(base):
        if name in fs:
            return os.path.join(r, name)
    return None

ckpt = find_file(f'{JOB}_best.pth')
meta_path = find_file(f'{JOB}_meta.json')
assert ckpt and meta_path, f'{JOB}_best.pth not found -- attach the training notebook that produced it'
meta = json.load(open(meta_path))
model = build_model(meta['variant'], NP_WEIGHTS)
sd = torch.load(ckpt, map_location='cpu', weights_only=False)
missing, unexpected = model.load_state_dict(sd, strict=False)
assert not missing and not unexpected, (missing[:5], unexpected[:5])
model = model.to(device).eval()
print(f"loaded {JOB}: variant {meta['variant']}, seed {meta['seed']}, best val F1 "
      f"{meta['best_val']['val_f1']:.4f} (epoch {meta['best_epoch']}), {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params")


## 4. Test sets

In [ ]:
# ── Dataset loaders: every entry is (image_path, mask_path_or_None, label, mask_kind) ──────
# mask_kind tells load_gt() how to turn the mask file into {0,1}:
#   'binary'   -> grayscale > 127 (CASIA v1/v2, IMD2020)
#   'columbia' -> colour-coded region map: tampered = pixels where the COLUMBIA_TAMPERED
#                 colour channel dominates (NOT grayscale>127, which keeps only the thin
#                 bright-green boundary band -- the bug behind the old Columbia numbers)
import os, glob, json, random
import numpy as np
import cv2

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
COLUMBIA_TAMPERED = 'green'   # verify against the README printed in the Columbia cell


def load_gt(mask_path, mask_kind, target_hw):
    """Returns a {0,1} uint8 mask at target_hw=(H, W) of the ORIGINAL image, or None."""
    if mask_path is None:
        return None
    if mask_kind == 'columbia':
        bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        g = bgr[..., 1].astype(np.int16)
        r = bgr[..., 2].astype(np.int16)
        m = (g > r) if COLUMBIA_TAMPERED == 'green' else (r > g)
        m = m.astype(np.uint8)
    else:
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            return None
        m = (m > 127).astype(np.uint8)
    if m.shape[:2] != tuple(target_hw):
        m = cv2.resize(m, (target_hw[1], target_hw[0]), interpolation=cv2.INTER_NEAREST)
    return m


# ── CASIA v2 in-distribution split ─────────────────────────────────────────────────────
# 'validated' = the split used by every RAFT-Net stage, MPC and LoMa (unreadable files such as
#               Thumbs.db dropped BEFORE shuffling -> 12472 samples -> 11225/1247).
# 'cfl'       = CFL-Net's split (nothing dropped -> 12473 samples incl. Thumbs.db -> 11226/1247).
# The two share only ~41% of their validation images, so CFL-Net must be scored on its OWN
# split (the other one contains ~59% of CFL-Net's training images).
def casia2_file_list(au_dir, tp_dir, gt_dir):
    samples = []
    for f in sorted(os.listdir(au_dir)):
        p = os.path.join(au_dir, f)
        if os.path.isfile(p) and not f.startswith('.'):
            samples.append((p, None))
    gt_by_stem = {}
    for f in os.listdir(gt_dir):
        if f.startswith('.'):
            continue
        stem = os.path.splitext(f)[0]
        if stem.endswith('_gt'):
            stem = stem[:-3]
        gt_by_stem[stem] = os.path.join(gt_dir, f)
    for f in sorted(os.listdir(tp_dir)):
        p = os.path.join(tp_dir, f)
        if not (os.path.isfile(p) and not f.startswith('.')):
            continue
        m = gt_by_stem.get(os.path.splitext(f)[0])
        if m is not None:
            samples.append((p, m))
    return samples


def casia2_val_split(au_dir, tp_dir, gt_dir, variant, seed=1221, val_fraction=0.1,
                     readable_check=None):
    samples = casia2_file_list(au_dir, tp_dir, gt_dir)
    if variant == 'validated':
        assert readable_check is not None
        samples = [s for s in samples if readable_check(s)]
    elif variant != 'cfl':
        raise ValueError(variant)
    random.Random(seed).shuffle(samples)
    n_val = int(len(samples) * val_fraction)
    val, train = samples[:n_val], samples[n_val:]
    # CFL's list can contain an unreadable file in val; it is skipped at load time exactly as
    # CFL-Net's own evaluation did (reported as skipped).
    as_entries = lambda lst: [(img, m, 0 if m is None else 1, 'binary') for img, m in lst]
    return as_entries(val), as_entries(train)


# ── Columbia ────────────────────────────────────────────────────────────────────────────
def columbia_entries(root):
    splc_img = [d for d in glob.glob(os.path.join(root, '**', '4cam_splc'), recursive=True)
                if any(f.lower().endswith(('.tif', '.tiff')) for f in os.listdir(d))]
    auth_img = [d for d in glob.glob(os.path.join(root, '**', '4cam_auth'), recursive=True)
                if any(f.lower().endswith(('.tif', '.tiff')) for f in os.listdir(d))]
    edge_dirs = [d for d in glob.glob(os.path.join(root, '**', 'edgemask'), recursive=True)
                 if '4cam_splc' in d]
    assert splc_img and auth_img and edge_dirs, (splc_img, auth_img, edge_dirs)
    edge = {}
    for d in edge_dirs:
        for f in os.listdir(d):
            stem = os.path.splitext(f)[0]
            if stem.endswith('_edgemask'):          # '_edgemask_3' variants are ignored
                edge[stem[:-len('_edgemask')]] = os.path.join(d, f)
    entries, unmatched = [], []
    for f in sorted(os.listdir(splc_img[0])):
        if not f.lower().endswith(('.tif', '.tiff')):
            continue
        stem = os.path.splitext(f)[0]
        m = edge.get(stem)
        if m is None:
            unmatched.append(f)
        entries.append((os.path.join(splc_img[0], f), m, 1, 'columbia'))
    for f in sorted(os.listdir(auth_img[0])):
        if f.lower().endswith(('.tif', '.tiff')):
            entries.append((os.path.join(auth_img[0], f), None, 0, 'binary'))
    return entries, unmatched


# ── IMD2020 ─────────────────────────────────────────────────────────────────────────────
def imd2020_entries(root):
    auth_dir, img_dir, mask_dir = (os.path.join(root, d) for d in ('Authentic', 'img', 'mask'))
    masks = {}
    for f in os.listdir(mask_dir):
        stem = os.path.splitext(f)[0]
        if stem.lower().endswith('_mask'):
            masks[stem[:-5]] = os.path.join(mask_dir, f)
    entries, unmatched = [], []
    for f in sorted(os.listdir(img_dir)):
        if not f.lower().endswith(IMG_EXTS):
            continue
        m = masks.get(os.path.splitext(f)[0])
        if m is None:
            unmatched.append(f)
        entries.append((os.path.join(img_dir, f), m, 1, 'binary'))
    for f in sorted(os.listdir(auth_dir)):
        if f.lower().endswith(IMG_EXTS):
            entries.append((os.path.join(auth_dir, f), None, 0, 'binary'))
    return entries, unmatched


# ── CASIA v1 ────────────────────────────────────────────────────────────────────────────
# The attached image mirror ships NO masks. Masks come from a separate groundtruth dataset
# (e.g. the community "CASIA 1.0 groundtruth" set: Sp_..._gt.png). Discovery is by NAME,
# not by folder layout: any image file under /kaggle/input whose stem (minus an optional
# '_gt' suffix) equals a CASIA1 tampered-image stem is taken as that image's mask.
def casia1_entries(casia1_root, search_root='/kaggle/input'):
    sp_dir = os.path.join(casia1_root, 'Sp')
    au_dir = os.path.join(casia1_root, 'Au')
    tamp = {os.path.splitext(f)[0]: os.path.join(sp_dir, f)
            for f in sorted(os.listdir(sp_dir)) if f.lower().endswith(IMG_EXTS)}
    masks = {}
    for r, dirs, files in os.walk(search_root):
        if os.path.abspath(r).startswith(os.path.abspath(sp_dir)):
            continue
        for f in files:
            if not f.lower().endswith(IMG_EXTS):
                continue
            stem = os.path.splitext(f)[0]
            base = stem[:-3] if stem.endswith('_gt') else stem
            if base in tamp and (stem.endswith('_gt') or 'truth' in r.lower() or 'mask' in r.lower()
                                 or 'gt' in os.path.basename(r).lower()):
                masks[base] = os.path.join(r, f)
    entries = [(p, masks.get(s), 1, 'binary') for s, p in tamp.items()]
    entries += [(os.path.join(au_dir, f), None, 0, 'binary')
                for f in sorted(os.listdir(au_dir)) if f.lower().endswith(IMG_EXTS)]
    return entries, sum(1 for s in tamp if s not in masks)


In [ ]:
# ── test sets (whichever are attached) ────────────────────────────────────────────────────────
def find_dirs(pattern):
    return sorted([p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) if os.path.isdir(p)], key=len)

def _norm(p):
    return p.lower().replace(' ', '').replace('_', '').replace('-', '')

def casia1_tampered(root='/kaggle/input'):
    tamp, masks = {}, {}
    for r, _, fs in os.walk(root):
        rn = _norm(r)
        gt_dir = ('truth' in rn) or ('mask' in rn)
        for f in fs:
            if not f.lower().endswith(IMG_EXTS):
                continue
            stem, p = os.path.splitext(f)[0], os.path.join(r, f)
            if stem.endswith('_gt') or (gt_dir and stem.startswith('Sp_')):
                masks.setdefault(stem[:-3] if stem.endswith('_gt') else stem, p)
            elif stem.startswith('Sp_') and not gt_dir:
                tamp.setdefault(stem, p)
    return [(tamp[s], masks[s], 1, 'binary') for s in sorted(tamp) if s in masks]

SETS = {}
sj = find_file('casia2_split_seed1221.json')
if sj:
    SETS['CASIA v2 (val)'] = [(a, b, 1, 'binary') for a, b in json.load(open(sj))['val'] if b]
c1 = casia1_tampered()
if c1:
    SETS['CASIA v1'] = c1
col_root = next((os.path.dirname(d) for d in find_dirs('4cam_splc') if os.path.isdir(os.path.join(os.path.dirname(d), '4cam_auth'))), None)
if col_root:
    SETS['Columbia'] = [e for e in columbia_entries(col_root)[0] if e[2] == 1 and e[1]]
imd_root = next((d for d in find_dirs('IMD2020') if all(os.path.isdir(os.path.join(d, s)) for s in ('Authentic', 'img', 'mask'))), None)
if imd_root:
    SETS['IMD2020'] = [e for e in imd2020_entries(imd_root)[0] if e[2] == 1 and e[1]]
for k, v in SETS.items():
    print(f'{k:<15} {len(v)} tampered images with masks')


## 5. Inference and drawing

In [ ]:
# ── inference + drawing ─────────────────────────────────────────────────────────────────────
@torch.no_grad()
def predict(bgr):
    h, w = bgr.shape[:2]
    x = cv2.resize(bgr, (INPUT, INPUT))[..., ::-1].astype(np.float32) / 255.0
    x = torch.from_numpy(np.ascontiguousarray(((x - MEAN) / STD).transpose(2, 0, 1)))[None].to(device)
    with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
        out = model(x)
    lg = (out['mask'] if isinstance(out, dict) else out[0]).float()
    d = F.interpolate(lg[:, 1:2] - lg[:, 0:1], (INPUT, INPUT), mode='bilinear', align_corners=True)
    prob = torch.sigmoid(d)[0, 0].cpu().numpy()
    return cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)

def f1_iou(prob, gt):
    p, g = prob > THR, gt > 0
    tp = (p & g).sum(); fp = (p & ~g).sum(); fn = (~p & g).sum()
    f1 = 2 * tp / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    return float(f1), float(iou)

def overlay(bgr, prob):
    rgb = bgr[..., ::-1].astype(np.float32) / 255.0
    m = (prob > THR)[..., None]
    tint = rgb * 0.45 + np.array([1.0, 0.1, 0.1]) * 0.55
    out = np.where(m, tint, rgb * 0.85)
    edge = cv2.morphologyEx((prob > THR).astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
    out[edge] = [1.0, 1.0, 0.0]
    return np.clip(out, 0, 1)

def draw_rows(rows, title, path):
    """rows: list of dicts with bgr, prob, gt (or None), label"""
    has_gt = any(r['gt'] is not None for r in rows)
    ncol = 5 if has_gt else 4
    fig, axes = plt.subplots(len(rows), ncol, figsize=(3.2 * ncol, 2.9 * len(rows)), squeeze=False)
    for i, r in enumerate(rows):
        panels = [('image', r['bgr'][..., ::-1])]
        if has_gt:
            panels.append(('ground truth', r['gt'] if r['gt'] is not None else np.zeros(r['bgr'].shape[:2])))
        panels += [('predicted mask', (r['prob'] > THR).astype(np.float32)),
                   ('tamper probability', r['prob']), ('prediction overlay', overlay(r['bgr'], r['prob']))]
        for j, (name, im) in enumerate(panels):
            ax = axes[i, j]
            if name == 'tamper probability':
                ax.imshow(im, cmap='inferno', vmin=0, vmax=1)
            elif name in ('ground truth', 'predicted mask'):
                ax.imshow(im, cmap='gray', vmin=0, vmax=1)
            else:
                ax.imshow(im)
            ax.set_xticks([]); ax.set_yticks([])
            if i == 0:
                ax.set_title(name, fontsize=10)
        axes[i, 0].set_ylabel(r['label'], fontsize=8)
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(path, dpi=110, bbox_inches='tight')
    plt.show()
print('ready')


## 6. Segment one image

In [ ]:
# ── 0. segment ONE image: the basic demo ────────────────────────────────────────────────────
IMAGE_PATH = None      # any image path; None = the first tampered image of the first attached test set
MASK_PATH = None       # optional ground-truth mask for that image

def segment(path, mask_path=None, kind='binary'):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    assert bgr is not None, f'cannot read {path}'
    prob = predict(bgr)
    pred = (prob > THR).astype(np.uint8)
    stem = os.path.splitext(os.path.basename(path))[0]
    os.makedirs(f'{OUT}/segmentations', exist_ok=True)
    cv2.imwrite(f'{OUT}/segmentations/{stem}_mask.png', pred * 255)                       # binary mask
    cv2.imwrite(f'{OUT}/segmentations/{stem}_overlay.png', (overlay(bgr, prob)[..., ::-1] * 255).astype(np.uint8))
    gt = load_gt(mask_path, kind, bgr.shape[:2]) if mask_path else None
    panels = [('input image', bgr[..., ::-1], None), ('predicted mask (white = tampered)', pred, 'gray'),
              ('tamper probability', prob, 'inferno'), ('overlay', overlay(bgr, prob), None)]
    if gt is not None:
        panels.insert(1, ('ground truth', gt, 'gray'))
    fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.6))
    for ax, (name, im, cmap) in zip(axes, panels):
        ax.imshow(im, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
        ax.set_title(name, fontsize=12); ax.set_xticks([]); ax.set_yticks([])
    txt = f'{stem}: {100 * pred.mean():.1f}% of pixels predicted tampered'
    if gt is not None:
        f1, iou = f1_iou(prob, gt)
        txt += f' | F1 {100 * f1:.1f}, IoU {100 * iou:.1f}'
    fig.suptitle(txt, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{OUT}/segmentations/{stem}_panel.png', dpi=110, bbox_inches='tight'); plt.show()
    print(txt, f'\nsaved: {OUT}/segmentations/{stem}_mask.png, _overlay.png, _panel.png')

if IMAGE_PATH:
    segment(IMAGE_PATH, MASK_PATH)
else:
    first = next(iter(SETS))
    ip, mp, _, kind = SETS[first][0]
    segment(ip, mp, kind)


## 7. Test-set showcase (best / typical / worst)

In [ ]:
# ── 1. test-set showcase: best / typical / worst of N_SAMPLE random tampered images per set ───
SUMMARY = {}
for name, ents in SETS.items():
    sample = random.Random(SEED).sample(ents, min(N_SAMPLE, len(ents)))
    scored = []
    for ip, mp, _, kind in sample:
        bgr = cv2.imread(ip, cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        gt = load_gt(mp, kind, bgr.shape[:2])
        if gt is None or gt.sum() == 0:
            continue
        prob = predict(bgr)
        f1, iou = f1_iou(prob, gt)
        scored.append(dict(bgr=bgr, gt=gt, prob=prob, f1=f1, iou=iou, file=os.path.basename(ip)))
    scored.sort(key=lambda r: -r['f1'])
    n = len(scored)
    picks = [('best', r) for r in scored[:SHOW['best']]]
    mid = n // 2
    picks += [('typical', r) for r in scored[max(0, mid - SHOW['typical'] // 2): mid - SHOW['typical'] // 2 + SHOW['typical']]]
    picks += [('worst', r) for r in scored[-SHOW['worst']:]]
    rows = [dict(bgr=r['bgr'], gt=r['gt'], prob=r['prob'],
                 label=f"{tag}: F1 {r['f1'] * 100:.0f} / IoU {r['iou'] * 100:.0f}\n{r['file'][:28]}") for tag, r in picks]
    fname = f"{OUT}/showcase_{name.split()[0].lower()}{'_v2' if 'v2' in name else ''}.png"
    draw_rows(rows, f"{name}: best, typical and worst of {n} random tampered images ({JOB})", fname)
    SUMMARY[name] = dict(n=n, mean_f1=float(np.mean([r['f1'] for r in scored])) * 100,
                         median_f1=float(np.median([r['f1'] for r in scored])) * 100)
    print(f"{name}: sample mean F1 {SUMMARY[name]['mean_f1']:.1f} | median {SUMMARY[name]['median_f1']:.1f} (n={n}) -> {fname}")


## 8. Robustness demo

In [ ]:
# ── 2. robustness demo: same image under compression ───────────────────────────────────────
def jpeg(img, q):
    ok, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, q])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR) if ok else img

def osn(img, scale=0.7, q=65):
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
    return cv2.resize(jpeg(small, q), (w, h), interpolation=cv2.INTER_LINEAR)

LEVELS = [('clean', lambda x: x), ('JPEG 90', lambda x: jpeg(x, 90)), ('JPEG 70', lambda x: jpeg(x, 70)),
          ('JPEG 50', lambda x: jpeg(x, 50)), ('OSN chain', osn)]
src_name = 'CASIA v1' if 'CASIA v1' in SETS else next(iter(SETS))
cands = random.Random(SEED + 1).sample(SETS[src_name], min(60, len(SETS[src_name])))
chosen = []
for ip, mp, _, kind in cands:                      # images the model localises well when clean
    bgr = cv2.imread(ip, cv2.IMREAD_COLOR)
    gt = load_gt(mp, kind, bgr.shape[:2]) if bgr is not None else None
    if gt is None or gt.sum() == 0:
        continue
    if f1_iou(predict(bgr), gt)[0] > 0.7:
        chosen.append((bgr, gt, os.path.basename(ip)))
    if len(chosen) == ROBUST_IMAGES:
        break
if not chosen:
    raise SystemExit('no well-localised image found for the robustness demo -- increase the candidate pool')
fig, axes = plt.subplots(len(chosen), len(LEVELS) + 1, figsize=(2.6 * (len(LEVELS) + 1), 2.5 * len(chosen)), squeeze=False)
for i, (bgr, gt, fn) in enumerate(chosen):
    axes[i, 0].imshow(gt, cmap='gray'); axes[i, 0].set_ylabel(fn[:22], fontsize=7)
    if i == 0:
        axes[i, 0].set_title('ground truth', fontsize=9)
    for j, (lv, fnc) in enumerate(LEVELS, start=1):
        img = fnc(bgr.copy())
        prob = predict(img)
        f1, _ = f1_iou(prob, gt)
        axes[i, j].imshow(overlay(img, prob))
        axes[i, j].set_title(f'{lv}\nF1 {f1 * 100:.0f}' if i == 0 else f'F1 {f1 * 100:.0f}', fontsize=9)
    for a in axes[i]:
        a.set_xticks([]); a.set_yticks([])
fig.suptitle(f'Robustness: predictions on the same images after compression ({src_name}, {JOB})', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUT}/robustness_demo.png', dpi=110, bbox_inches='tight'); plt.show()


## 9. Your own images

In [ ]:
# ── 3. your own images (no ground truth needed) ─────────────────────────────────────────────
udir = USER_IMAGE_DIR or next((d for d in glob.glob('/kaggle/input/**/*', recursive=True)
                               if os.path.isdir(d) and 'demo' in os.path.basename(d).lower()), None)
if udir:
    files = sorted(f for f in glob.glob(f'{udir}/**/*', recursive=True) if f.lower().endswith(IMG_EXTS))[:12]
    rows = []
    for fp in files:
        bgr = cv2.imread(fp, cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        prob = predict(bgr)
        rows.append(dict(bgr=bgr, gt=None, prob=prob,
                         label=f"{os.path.basename(fp)[:26]}\ntampered area {100 * (prob > THR).mean():.1f}%"))
    if rows:
        draw_rows(rows, f'Your images ({JOB})', f'{OUT}/user_images.png')
else:
    print('No user images attached (add a dataset whose name contains "demo", or set USER_IMAGE_DIR).')
print('\nSaved figures:', sorted(os.listdir(OUT)))
print('Sample statistics:', json.dumps(SUMMARY, indent=1))


## 10. Efficiency

In [ ]:
# ── 10. efficiency: parameters, inference latency and peak GPU memory (512x512, batch 1) ─────
import time

def bench(variant, n_warm=10, n_iter=50):
    m = build_model(variant, NP_WEIGHTS).to(device).eval()
    params_total = sum(p.numel() for p in m.parameters()) / 1e6
    if getattr(m, 'boundary', None) is not None:
        m.boundary = None                      # training-only branch: not used at inference
    params_inf = sum(p.numel() for p in m.parameters()) / 1e6
    x = torch.randn(1, 3, INPUT, INPUT, device=device)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
        for _ in range(n_warm):
            m(x)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter):
            m(x)
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / n_iter * 1000
    mem = torch.cuda.max_memory_allocated() / 2**20
    del m; torch.cuda.empty_cache()
    return params_total, params_inf, ms, mem

EFF = {}
for name, variant in [('ForMa', 'forma'), ('RAFT-Net (final)', 'raftnet_lite_nofullres'), ('RAFT-Net, full-res residuals', 'raftnet_no_fcss'), ('RAFT-Net + FCSS', 'raftnet')]:
    pt, pi, ms, mem = bench(variant)
    EFF[name] = dict(params_total_m=pt, params_inference_m=pi, latency_ms=ms, images_per_s=1000 / ms, peak_mem_mb=mem)
    print(f"{name:<18} params {pt:6.2f} M (at inference {pi:6.2f} M) | {ms:6.1f} ms/image | {1000 / ms:5.1f} img/s | "
          f"peak GPU memory {mem:6.0f} MB")
json.dump(EFF, open(f'{OUT}/efficiency.json', 'w'), indent=1)
print('GPU:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu', '| saved', f'{OUT}/efficiency.json')


## What to send back
The PNG files in `/kaggle/working/demo/` (showcase per dataset, robustness demo, your images). They go into the
report's qualitative case-study section.
